In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is not set")

os.environ["GOOGLE_API_KEY"] = google_api_key

In [5]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
! pip install langchain-chroma

  Obtaining dependency information for langchain-chroma from https://files.pythonhosted.org/packages/ae/35/2a6d1191acaad043647e28313b0ecd161d61f09d8be37d1996a90d752c13/langchain_chroma-1.1.0-py3-none-any.whl.metadata

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

In [10]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [11]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [15]:
vector_store = Chroma(
    embedding_function=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", output_dimensionality=32),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [16]:
# add documents
vector_store.add_documents(docs)

['7bdebdc1-e532-405b-a1dc-f366dfce5a29',
 '6adc658b-be55-42ea-a441-27736ee0a30b',
 '837b0a37-8240-4d5d-8d77-dd553e7b3176',
 '01ecb94d-1afb-40dc-bdda-21780db82bde',
 'b4bd3367-464e-4000-aa61-9463e4c5224c']

In [17]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['7bdebdc1-e532-405b-a1dc-f366dfce5a29',
  '6adc658b-be55-42ea-a441-27736ee0a30b',
  '837b0a37-8240-4d5d-8d77-dd553e7b3176',
  '01ecb94d-1afb-40dc-bdda-21780db82bde',
  'b4bd3367-464e-4000-aa61-9463e4c5224c'],
 'embeddings': array([[-9.82054323e-03,  2.54576281e-02,  2.40278151e-02,
         -6.40832633e-02, -3.31680733e-03,  2.58310558e-03,
          1.69644167e-03,  2.30533648e-02, -4.09102254e-02,
         -4.66324818e-05,  4.45438828e-03, -1.35538634e-02,
         -5.71147818e-03,  6.76856264e-02,  1.08889811e-01,
         -2.20292388e-03, -2.19895714e-03, -1.68876257e-02,
          4.12407098e-03, -3.92815704e-03, -5.10695251e-03,
         -1.01319300e-02, -1.36001821e-04, -1.67653505e-02,
         -5.70380874e-03, -2.11964808e-02,  2.28025876e-02,
          1.07095717e-02,  1.67755075e-02,  6.69531804e-03,
         -1.50224362e-02, -6.68604998e-03],
        [-1.98972058e-02,  1.09271351e-02,  1.73060261e-02,
         -6.74819872e-02,  7.47970422e-04,  6.26170635e-03,
    

In [18]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='b4bd3367-464e-4000-aa61-9463e4c5224c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
 Document(id='01ecb94d-1afb-40dc-bdda-21780db82bde', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [19]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='b4bd3367-464e-4000-aa61-9463e4c5224c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.007737690582871437),
 (Document(id='01ecb94d-1afb-40dc-bdda-21780db82bde', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.009246289730072021)]

In [21]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [22]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['7bdebdc1-e532-405b-a1dc-f366dfce5a29',
  '6adc658b-be55-42ea-a441-27736ee0a30b',
  '837b0a37-8240-4d5d-8d77-dd553e7b3176',
  '01ecb94d-1afb-40dc-bdda-21780db82bde',
  'b4bd3367-464e-4000-aa61-9463e4c5224c'],
 'embeddings': array([[-9.82054323e-03,  2.54576281e-02,  2.40278151e-02,
         -6.40832633e-02, -3.31680733e-03,  2.58310558e-03,
          1.69644167e-03,  2.30533648e-02, -4.09102254e-02,
         -4.66324818e-05,  4.45438828e-03, -1.35538634e-02,
         -5.71147818e-03,  6.76856264e-02,  1.08889811e-01,
         -2.20292388e-03, -2.19895714e-03, -1.68876257e-02,
          4.12407098e-03, -3.92815704e-03, -5.10695251e-03,
         -1.01319300e-02, -1.36001821e-04, -1.67653505e-02,
         -5.70380874e-03, -2.11964808e-02,  2.28025876e-02,
          1.07095717e-02,  1.67755075e-02,  6.69531804e-03,
         -1.50224362e-02, -6.68604998e-03],
        [-1.98972058e-02,  1.09271351e-02,  1.73060261e-02,
         -6.74819872e-02,  7.47970422e-04,  6.26170635e-03,
    

In [23]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [24]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['7bdebdc1-e532-405b-a1dc-f366dfce5a29',
  '6adc658b-be55-42ea-a441-27736ee0a30b',
  '837b0a37-8240-4d5d-8d77-dd553e7b3176',
  '01ecb94d-1afb-40dc-bdda-21780db82bde',
  'b4bd3367-464e-4000-aa61-9463e4c5224c'],
 'embeddings': array([[-9.82054323e-03,  2.54576281e-02,  2.40278151e-02,
         -6.40832633e-02, -3.31680733e-03,  2.58310558e-03,
          1.69644167e-03,  2.30533648e-02, -4.09102254e-02,
         -4.66324818e-05,  4.45438828e-03, -1.35538634e-02,
         -5.71147818e-03,  6.76856264e-02,  1.08889811e-01,
         -2.20292388e-03, -2.19895714e-03, -1.68876257e-02,
          4.12407098e-03, -3.92815704e-03, -5.10695251e-03,
         -1.01319300e-02, -1.36001821e-04, -1.67653505e-02,
         -5.70380874e-03, -2.11964808e-02,  2.28025876e-02,
          1.07095717e-02,  1.67755075e-02,  6.69531804e-03,
         -1.50224362e-02, -6.68604998e-03],
        [-1.98972058e-02,  1.09271351e-02,  1.73060261e-02,
         -6.74819872e-02,  7.47970422e-04,  6.26170635e-03,
    